<a href="https://colab.research.google.com/github/victoria-analyst/Exploratory-Data-Analysis-for-an-International-E-commerce-Company/blob/main/Project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Code

### Data Import

In [ ]:
!pip install --upgrade google-cloud-bigquery

In [ ]:
from google.colab import auth
from google.cloud import bigquery
from google.colab import drive

from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

import numpy as np
import pandas as pd

auth.authenticate_user()

In [ ]:
client = bigquery.Client(project="data-analytics-mate")
query = """
WITH
  session_info AS (
    SELECT
      s.date,
      s.ga_session_id,
      sp.country,
      sp.device,
      sp.continent,
      sp.channel,
      ab.test,
      ab.test_group
    FROM `DA.ab_test` ab
    JOIN `DA.session` s
      ON ab.ga_session_id = s.ga_session_id
    JOIN `DA.session_params` sp
      ON s.ga_session_id = sp.ga_session_id
  ),

  session_with_orders AS (
    SELECT
      session_info.date,
      session_info.ga_session_id,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      COUNT(DISTINCT o.ga_session_id) AS session_with_orders
    FROM session_info
    JOIN `DA.order` o
      ON session_info.ga_session_id = o.ga_session_id
    GROUP BY
      session_info.date,
      session_info.ga_session_id,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group
  ),

  events AS (
    SELECT
      session_info.date,
      session_info.ga_session_id,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      ep.event_name,
      COUNT(ep.ga_session_id) AS events_cnt
    FROM session_info
    JOIN `DA.event_params` ep
      ON session_info.ga_session_id = ep.ga_session_id
    GROUP BY
      session_info.date,
      session_info.ga_session_id,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      ep.event_name
  ),

  session AS (
    SELECT
      session_info.date,
      session_info.ga_session_id,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      COUNT(DISTINCT session_info.ga_session_id) AS sessions_cnt
    FROM session_info
    GROUP BY
      session_info.date,
      session_info.ga_session_id,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group
  ),

  account AS (
    SELECT
      session_info.date,
      session_info.ga_session_id,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      COUNT(acs.ga_session_id) AS new_accounts_cnt
    FROM session_info
    JOIN `DA.account_session` acs
      ON session_info.ga_session_id = acs.ga_session_id
    GROUP BY
      session_info.date,
      session_info.ga_session_id,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group
  )


SELECT
  session_with_orders.date,
  session_with_orders.ga_session_id,
  session_with_orders.country,
  session_with_orders.device,
  session_with_orders.continent,
  session_with_orders.channel,
  session_with_orders.test,
  session_with_orders.test_group,
  "session with orders" AS event_name,
  session_with_orders.session_with_orders AS value
FROM session_with_orders

UNION ALL

SELECT
    events.date,
    events.ga_session_id,
    events.country,
    events.device,
    events.continent,
    events.channel,
    events.test,
    events.test_group,
    events.event_name,
    events.events_cnt as value
FROM events

UNION ALL

SELECT
    session.date,
    session.ga_session_id,
    session.country,
    session.device,
    session.continent,
    session.channel,
    session.test,
    session.test_group,
    'session' AS event_name,
    session.sessions_cnt as value
FROM session

UNION ALL

SELECT
    account.date,
    account.ga_session_id,
    account.country,
    account.device,
    account.continent,
    account.channel,
    account.test,
    account.test_group,
    'new account' AS event_name,
    account.new_accounts_cnt as value
FROM account
"""
query_job = client.query(query)
result = query_job.result()

In [ ]:
data = result.to_dataframe()
data.head()

,date,ga_session_id,country,device,continent,channel,test,test_group,event_name,value
0,2020-11-02,7228028233,New Zealand,desktop,Oceania,Organic Search,2,1,session with orders,1
1,2020-11-07,4038354492,Slovenia,mobile,Europe,Organic Search,2,2,session with orders,1
2,2020-11-08,4229611646,Colombia,mobile,Americas,Paid Search,2,1,session with orders,1
3,2020-11-16,7033745481,Serbia,mobile,Europe,Organic Search,2,1,session with orders,1
4,2020-11-19,1266713490,Austria,desktop,Europe,Organic Search,2,1,session with orders,1


In [ ]:
data["event_name"].unique()

array(['session with orders', 'new account', 'session', 'session_start',
       'page_view', 'view_item', 'user_engagement', 'view_promotion',
       'scroll', 'view_search_results', 'first_visit', 'select_item',
       'add_to_cart', 'add_payment_info', 'select_promotion',
       'add_shipping_info', 'begin_checkout', 'click', 'view_item_list'],
      dtype=object)

### Calculation by Test

In [ ]:
metrics = ["add_payment_info", "add_shipping_info", "begin_checkout", "new account"]
results = []

for test_name, test_data in data.groupby("test"):

    sessions = test_data.groupby("test_group")["ga_session_id"].nunique()

    for metric in metrics:
        event_sessions = test_data[test_data["event_name"] == metric].groupby("test_group")["value"].sum()

        numerator_A = event_sessions.get(1, 0)
        numerator_B = event_sessions.get(2, 0)

        denominator_A = sessions.get(1, 0)
        denominator_B = sessions.get(2, 0)

        if denominator_A == 0 or denominator_B == 0:
          continue

        event_counts = np.array([numerator_A, numerator_B])
        sessions_counts = np.array([denominator_A, denominator_B])

        if numerator_A == 0 and numerator_B == 0:
            p_value = np.nan; stat = np.nan
        else:
            stat, p_value = proportions_ztest(event_counts, sessions_counts)

        conv_A = numerator_A / denominator_A
        conv_B = numerator_B / denominator_B

        uplift = (conv_B - conv_A) / conv_A

        results.append({
            "test": test_name,
            "metric": metric,
            "sessions_A": denominator_A,
            "sessions_B": denominator_B,
            "events_A": numerator_A,
            "events_B": numerator_B,
            "conversion_A": conv_A,
            "conversion_B": conv_B,
            "uplift": uplift,
            "p_value": p_value if not np.isnan(p_value) else None,
            "significant": p_value < 0.05,
            "z_stat": stat
        })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(["test", "metric"])

results_df.round(5)

,test,metric,sessions_A,sessions_B,events_A,events_B,conversion_A,conversion_B,uplift,p_value,significant,z_stat
0,1,add_payment_info,45362,45193,1988,2229,0.04383,0.04932,0.12542,0.00009,True,-3.92488
1,1,add_shipping_info,45362,45193,3034,3221,0.06688,0.07127,0.06560,0.00923,True,-2.60357
2,1,begin_checkout,45362,45193,3784,4021,0.08342,0.08897,0.06661,0.00289,True,-2.97878
3,1,new account,45362,45193,3823,3681,0.08428,0.08145,-0.03354,0.12286,False,1.54288
4,2,add_payment_info,50637,50244,2344,2409,0.04629,0.04795,0.03577,0.21461,False,-1.24099
5,2,add_shipping_info,50637,50244,3480,3510,0.06872,0.06986,0.01651,0.47798,False,-0.70956
6,2,begin_checkout,50637,50244,4262,4313,0.08417,0.08584,0.01988,0.34064,False,-0.95290
7,2,new account,50637,50244,4165,4184,0.08225,0.08327,0.01242,0.55600,False,-0.58879
8,3,add_payment_info,70047,70439,3623,3697,0.05172,0.05249,0.01475,0.52011,False,-0.64317
9,3,add_shipping_info,70047,70439,5298,5188,0.07563,0.07365,-0.02621,0.15744,False,1.41373


###CSV Export

In [ ]:
# export
drive.mount("/content/drive", force_remount=True)
results_df.to_csv("ab_test_results_2.csv", index=False, sep=";")
!cp ab_test_results_2.csv "/content/drive/MyDrive/Portfolio/"

Mounted at /content/drive


https://drive.google.com/file/d/1FIt5X0dNxO_1B6xfdKQ2ZFfE-0bTQBD1/view?usp=sharing

### Calculation by Test, Device

In [ ]:
metrics = ["add_payment_info", "add_shipping_info", "begin_checkout", "new account"]
results_device = []

for device, device_data in data.groupby("device"):
    for test_name, test_data in device_data.groupby("test"):
        sessions = test_data.groupby("test_group")["ga_session_id"].nunique()

        for metric in metrics:
            event_sessions = test_data[test_data["event_name"] == metric].groupby("test_group")["value"].sum()

            numerator_A = event_sessions.get(1, 0)
            numerator_B = event_sessions.get(2, 0)

            denominator_A = sessions.get(1, 0)
            denominator_B = sessions.get(2, 0)

            if denominator_A == 0 or denominator_B == 0:
              continue

            event_counts = np.array([numerator_A, numerator_B])
            sessions_counts = np.array([denominator_A, denominator_B])

            if numerator_A == 0 and numerator_B == 0:
                p_value = np.nan
            else:
                stat, p_value = proportions_ztest(event_counts, sessions_counts)

            conv_A = numerator_A / denominator_A
            conv_B = numerator_B / denominator_B

            uplift = (conv_B - conv_A) / conv_A

            results_device.append({
                "device": device,
                "test": test_name,
                "metric": metric,
                "sessions_A": denominator_A,
                "sessions_B": denominator_B,
                "events_A": numerator_A,
                "events_B": numerator_B,
                "conversion_A": conv_A,
                "conversion_B": conv_B,
                "uplift": uplift,
                "p_value": p_value,
                "significant": p_value < 0.05,
                "z_stat": stat
            })

results_device_df = pd.DataFrame(results_device)
results_device_df = results_device_df.sort_values(["device", "test", "metric"])
results_device_df

,device,test,metric,sessions_A,sessions_B,events_A,events_B,conversion_A,conversion_B,uplift,p_value,significant,z_stat
0,desktop,1,add_payment_info,26467,26417,1130,1256,0.042695,0.047545,0.113608,0.007210,True,-2.686998
1,desktop,1,add_shipping_info,26467,26417,1711,1916,0.064647,0.072529,0.121932,0.000336,True,-3.586024
2,desktop,1,begin_checkout,26467,26417,2108,2404,0.079646,0.091002,0.142576,0.000003,True,-4.673980
3,desktop,1,new account,26467,26417,2203,2147,0.083236,0.081273,-0.023575,0.411526,False,0.821212
4,desktop,2,add_payment_info,29497,29380,1314,1401,0.044547,0.047686,0.070456,0.069434,False,-1.815587
5,desktop,2,add_shipping_info,29497,29380,1988,2088,0.067397,0.071069,0.054484,0.079252,False,-1.755041
6,desktop,2,begin_checkout,29497,29380,2403,2585,0.081466,0.087985,0.080023,0.004507,True,-2.840291
7,desktop,2,new account,29497,29380,2442,2385,0.082788,0.081178,-0.019452,0.476356,False,0.712176
8,desktop,3,add_payment_info,41106,41443,2087,2167,0.050771,0.052289,0.029889,0.324108,False,-0.986051
9,desktop,3,add_shipping_info,41106,41443,3120,3072,0.075901,0.074126,-0.023391,0.332911,False,0.968268


### Calculation by Test, Continent

In [ ]:
metrics = ["add_payment_info", "add_shipping_info", "begin_checkout", "new account"]
results_continent = []

for continent, continent_data in data.groupby("continent"):
    for test_name, test_data in device_data.groupby("test"):
        sessions = test_data.groupby("test_group")["ga_session_id"].nunique()

        for metric in metrics:
            event_sessions = test_data[test_data["event_name"] == metric].groupby("test_group")["value"].sum()

            numerator_A = event_sessions.get(1, 0)
            numerator_B = event_sessions.get(2, 0)

            denominator_A = sessions.get(1, 0)
            denominator_B = sessions.get(2, 0)

            if denominator_A == 0 or denominator_B == 0:
              continue

            event_counts = np.array([numerator_A, numerator_B])
            sessions_counts = np.array([denominator_A, denominator_B])

            if numerator_A == 0 and numerator_B == 0:
                p_value = np.nan
            else:
                stat, p_value = proportions_ztest(event_counts, sessions_counts)

            conv_A = numerator_A / denominator_A
            conv_B = numerator_B / denominator_B

            uplift = (conv_B - conv_A) / conv_A

            results_continent.append({
                "continent": continent,
                "test": test_name,
                "metric": metric,
                "sessions_A": denominator_A,
                "sessions_B": denominator_B,
                "events_A": numerator_A,
                "events_B": numerator_B,
                "conversion_A": conv_A,
                "conversion_B": conv_B,
                "uplift": uplift,
                "p_value": p_value,
                "significant": p_value < 0.05,
                "z_stat": stat
            })

results_continent_df = pd.DataFrame(results_continent)
results_continent_df = results_continent_df.sort_values(["continent", "test", "metric"])
results_continent_df

,continent,test,metric,sessions_A,sessions_B,events_A,events_B,conversion_A,conversion_B,uplift,p_value,significant,z_stat
0,(not set),1,add_payment_info,999,1009,48,31,0.048048,0.030723,-0.360567,0.045868,True,1.996608
1,(not set),1,add_shipping_info,999,1009,66,49,0.066066,0.048563,-0.264934,0.091464,False,1.687725
2,(not set),1,begin_checkout,999,1009,83,56,0.083083,0.055500,-0.331988,0.014907,True,2.434631
3,(not set),1,new account,999,1009,90,93,0.090090,0.092170,0.023092,0.871341,False,-0.161955
4,(not set),2,add_payment_info,1123,1108,52,47,0.046305,0.042419,-0.083918,0.655864,False,0.445631
...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,Oceania,3,new account,1537,1573,126,137,0.081978,0.087095,0.062417,0.608121,False,-0.512758
92,Oceania,4,add_payment_info,2379,2360,107,54,0.044977,0.022881,-0.491264,0.000027,True,4.198074
93,Oceania,4,add_shipping_info,2379,2360,125,88,0.052543,0.037288,-0.290332,0.011267,True,2.534313
94,Oceania,4,begin_checkout,2379,2360,311,232,0.130727,0.098305,-0.248014,0.000459,True,3.503646


##Tableau Dashboards

[AB-test overview](https://public.tableau.com/views/ABtestUPDATED/ABtest?:language=en-GB&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link)

[Key metrics results](https://public.tableau.com/views/ABtestUPDATED/KeyMetrics?:language=en-GB&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link)

You can also switch between dashboards by clicking the up navigation button.